# Step 1 — Explore the GenoPHI E. coli interaction matrix

Goal: understand the labels before training any model. We inspect matrix size, class balance, phage host-range variation, and strain susceptibility.

First run `python scripts/fetch_genophi.py` from the repository root.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from phage_cocktail.analysis import summarize_interactions
from phage_cocktail.data import load_interactions

DATA_PATH = Path('../data/raw/genophi_ecoli_subset.csv')
df = load_interactions(DATA_PATH)
df.head()

In [ ]:
summary = summarize_interactions(df)
pd.Series(summary)

## Class balance

A naive accuracy score will be misleading because negative interactions are much more common than positive ones. Later we should emphasize AUROC, average precision / PR-AUC, calibration, and held-out-strain performance.

In [ ]:
class_counts = df['infection'].value_counts().sort_index()
ax = class_counts.rename(index={0: 'no infection', 1: 'infection'}).plot(kind='bar')
ax.set_ylabel('interaction count')
ax.set_title('GenoPHI E. coli subset — class balance')
plt.tight_layout()
plt.show()

## How broad is each phage's host range?

For each phage, calculate the fraction of the 177 strains it infects. This is a phenotype-only view; it does not yet use genome features.

In [ ]:
phage_hit_rate = (
    df.groupby('phage')['infection']
      .agg(['mean', 'sum', 'count'])
      .rename(columns={'mean': 'hit_rate', 'sum': 'infected_strains'})
      .sort_values('hit_rate', ascending=False)
)
phage_hit_rate.head(10)

In [ ]:
ax = phage_hit_rate['hit_rate'].plot(kind='hist', bins=20)
ax.set_xlabel('fraction of strains infected')
ax.set_title('Distribution of phage host-range breadth')
plt.tight_layout()
plt.show()

## How susceptible is each bacterial strain?

For each strain, calculate the fraction of the 94 phages that infect it. Large variation here means strain identity matters strongly and reinforces the need for strain-held-out evaluation.

In [ ]:
strain_susceptibility = (
    df.groupby('bacterium')['infection']
      .agg(['mean', 'sum', 'count'])
      .rename(columns={'mean': 'susceptibility', 'sum': 'infecting_phages'})
      .sort_values('susceptibility', ascending=False)
)
strain_susceptibility.head(10)

In [ ]:
ax = strain_susceptibility['susceptibility'].plot(kind='hist', bins=20)
ax.set_xlabel('fraction of phages that infect the strain')
ax.set_title('Distribution of bacterial susceptibility')
plt.tight_layout()
plt.show()

## What this tells us before modeling

1. The matrix is complete, so missing labels are not a problem in this first dataset.
2. Positive interactions are the minority class, so accuracy alone is not sufficient.
3. Phages differ in host-range breadth; a strong baseline must beat simply choosing the most promiscuous phage.
4. The target task is generalization to unseen bacterial strains, so later train/test splits must be grouped by bacterium.